# CloudCompute: inference
Запуск чемпиона или ансамбля на `test.zip` из локального постоянного каталога.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
MODEL = "best"  # best | mobilenet_v3_large | efficientnet_b0 | vit_b_16 | best_ensemble
BATCH_SIZE = None
NUM_WORKERS = 2
RESUME_INFERENCE = True
SMOKE_IMAGES = None
CONTACT_SHEET_COUNT = 16
RUN_TESTS = False
REPO_DIR = "/root/text-orientation-classification"
STATE_DIR = "/root/text-orientation-state"

In [ ]:
import os, subprocess, sys
from pathlib import Path
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir(): subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if RUN_TESTS: subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
test_zip = Path(STATE_DIR) / "data/test.zip"
registry = Path(STATE_DIR) / "registry"
if not test_zip.is_file(): raise FileNotFoundError(f"Missing {test_zip}")
if not (registry / "leaderboard.json").is_file(): raise FileNotFoundError(f"Missing registry in {registry}")

In [ ]:
command = [sys.executable, "-m", "scripts.infer", "--project-dir", STATE_DIR, "--model", MODEL, "--num-workers", str(NUM_WORKERS), "--contact-sheet-count", str(CONTACT_SHEET_COUNT)]
if BATCH_SIZE is not None: command += ["--batch-size", str(BATCH_SIZE)]
if SMOKE_IMAGES is not None: command += ["--limit", str(SMOKE_IMAGES)]
if not RESUME_INFERENCE: command.append("--no-resume")
subprocess.run(command, check=True)
runs = sorted((Path(STATE_DIR) / "inference/runs").glob("*"), key=lambda p: p.stat().st_mtime)
latest = runs[-1]
print("Result directory:", latest)
print("Submission:", latest / "submission.csv")